In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 และ Qiskit SDK

<details>
<summary><b>เวอร์ชันของ package</b></summary>

โค้ดในหน้านี้ถูกพัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK มีเครื่องมือบางอย่างสำหรับการแปลงระหว่าง OpenQASM representation ของโปรแกรม quantum และ [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) class

<span id="qasm2-import"></span>
## นำเข้าโปรแกรม OpenQASM 2 เข้าสู่ Qiskit
มีสองฟังก์ชันสำหรับนำเข้าโปรแกรม OpenQASM 2 เข้าสู่ Qiskit
ฟังก์ชันเหล่านี้คือ [`qasm2.load()`](../api/qiskit/qasm2#load) ซึ่งรับชื่อไฟล์ และ [`qasm2.loads()`](../api/qiskit/qasm2#loads) ซึ่งรับโปรแกรม OpenQASM 2 เป็น string

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

ดูข้อมูลเพิ่มเติมที่ [OpenQASM 2 Qiskit API](https://docs.quantum.ibm.com/api/qiskit/qasm2)

### นำเข้าโปรแกรมพื้นฐาน
สำหรับโปรแกรม OpenQASM 2 ส่วนใหญ่ สามารถใช้ `qasm2.load` และ `qasm2.loads` ด้วย argument เดียวได้

#### ตัวอย่าง: นำเข้าโปรแกรม OpenQASM 2 เป็น string
ใช้ `qasm2.loads()` เพื่อนำเข้าโปรแกรม OpenQASM 2 เป็น string เข้าสู่ QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### ตัวอย่าง: นำเข้าโปรแกรม OpenQASM 2 จากไฟล์
ใช้ `load()` เพื่อนำเข้าโปรแกรม OpenQASM 2 จากไฟล์เข้าสู่ QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### เชื่อมโยง Gate ของ OpenQASM 2 กับ Gate ของ Qiskit
โดยค่าเริ่มต้น importer ของ OpenQASM 2 ใน Qiskit จะถือว่าไฟล์ include `"qelib1.inc"` เป็น *de facto* standard library
importer ถือว่าไฟล์นี้ประกอบด้วย Gate ที่อธิบายไว้ใน [paper ต้นฉบับที่นิยาม OpenQASM 2](https://arxiv.org/abs/1707.03429) อย่างแม่นยำ
Qiskit จะใช้ Gate ที่มีอยู่แล้วใน [circuit library](../api/qiskit/circuit_library) เพื่อแทน Gate ใน `"qelib1.inc"`
Gate ที่นิยามในโปรแกรมด้วย OpenQASM 2 `gate` statement โดยค่าเริ่มต้นจะถูกสร้างเป็น [Qiskit `Gate` subclass](../api/qiskit/qiskit.circuit.Gate) แบบกำหนดเอง

สามารถบอก importer ให้ใช้ [`Gate`](../api/qiskit/qiskit.circuit.Gate) class ที่เฉพาะเจาะจงสำหรับ `gate` statement ที่กำหนดได้
สามารถใช้กลไกนี้เพื่อจัดการ gate name เพิ่มเติมว่าเป็น "built-in" นั่นคือไม่ต้องมี definition ที่ชัดเจน
หากระบุ Gate class ที่จะใช้สำหรับ `gate` statement ที่อยู่นอก `"qelib1.inc"` Circuit ที่ได้จะมีประสิทธิภาพมากขึ้นในการทำงาน

> **Warning:** ณ Qiskit SDK v1.0 *exporter* OpenQASM 2 ของ Qiskit (ดู [Export a Qiskit circuit to OpenQASM 2](#qasm2-export)) ยังคงทำงานราวกับว่า `"qelib1.inc"` มี Gate มากกว่าความเป็นจริง
> ซึ่งหมายความว่าการตั้งค่าเริ่มต้นของ importer อาจไม่สามารถนำเข้าโปรแกรมที่ export โดย importer ของเราได้
> ดู [ตัวอย่างการทำงานกับ legacy exporter](#qasm2-import-legacy) เพื่อแก้ปัญหานี้
> 
> ความแตกต่างนี้เป็น legacy behavior ของ Qiskit และ [จะถูกแก้ไขใน release ถัดไปของ Qiskit](https://github.com/Qiskit/qiskit/issues/10737)

หากต้องการส่งข้อมูลเกี่ยวกับ custom instruction ไปยัง OpenQASM 2 importer ให้ใช้ [คลาส `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction)
ต้องการข้อมูลสี่ส่วนตามลำดับ:

* **ชื่อ** ของ Gate ที่ใช้ในโปรแกรม OpenQASM 2
* **จำนวน angle parameter** ที่ Gate รับ
* **จำนวน Qubit** ที่ Gate ทำงานบน
* Python **constructor** class หรือฟังก์ชันของ Gate ซึ่งรับ Gate parameter (แต่ไม่ใช่ Qubit) เป็น argument แต่ละตัว

หาก importer พบ `gate` definition ที่ตรงกับ custom instruction ที่กำหนด จะใช้ข้อมูล custom นั้นเพื่อสร้าง Gate object ขึ้นใหม่
หากพบ `gate` statement ที่ตรงกับ `name` ของ custom instruction แต่ไม่ตรงกับทั้งจำนวน parameter และจำนวน Qubit importer จะ raise [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) เพื่อระบุความไม่ตรงกันระหว่างข้อมูลที่ให้มาและโปรแกรม

นอกจากนี้ argument ที่ห้า `builtin` สามารถตั้งค่าเป็น `True` เพื่อทำให้ Gate พร้อมใช้งานโดยอัตโนมัติภายในโปรแกรม OpenQASM 2 แม้ว่าจะไม่ได้นิยามอย่างชัดเจน
หาก importer พบ `gate` definition ที่ชัดเจนสำหรับ built-in custom instruction จะยอมรับอย่างเงียบๆ
เช่นเดิม หากพบ definition ที่ชัดเจนของชื่อเดียวกันแต่ไม่เข้ากันกับ custom instruction ที่ให้มา จะ raise [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror)
สิ่งนี้มีประโยชน์สำหรับความเข้ากันได้กับ OpenQASM 2 exporter รุ่นเก่า และกับ quantum platform บางแห่งที่ถือว่า "basis gate" ของฮาร์ดแวร์เป็น built-in instruction

Qiskit มี data attribute สำหรับการทำงานกับโปรแกรม OpenQASM 2 ที่สร้างโดย [Qiskit's OpenQASM 2 exporting capabilities](#qasm2-export) รุ่นเก่า
นั่นคือ [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) ซึ่งสามารถให้เป็น argument `custom_instructions` ของ [`qasm2.load()`](../api/qiskit/qasm2#load) และ [`qasm2.loads()`](../api/qiskit/qasm2#loads)

<span id="qasm2-import-legacy"></span>
#### ตัวอย่าง: นำเข้าโปรแกรมที่สร้างโดย legacy exporter ของ Qiskit
โปรแกรม OpenQASM 2 นี้ใช้ Gate ที่ไม่ได้อยู่ในเวอร์ชันต้นฉบับของ `"qelib1.inc"` โดยไม่มีการประกาศ แต่เป็น Gate มาตรฐานใน Qiskit library
สามารถใช้ [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) เพื่อบอก importer ให้ใช้ Gate set เดียวกับที่ OpenQASM 2 exporter ของ Qiskit เคยใช้ได้อย่างง่ายดาย

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### ตัวอย่าง: ใช้ Gate class ที่เฉพาะเจาะจงเมื่อนำเข้าโปรแกรม OpenQASM 2
Qiskit ไม่สามารถตรวจสอบได้โดยทั่วไปว่า definition ใน OpenQASM 2 `gate` statement ตรงกับ Qiskit standard-library gate อย่างแม่นยำหรือไม่
แทนที่ Qiskit จะเลือก custom Gate โดยใช้ definition ที่ให้มาอย่างแม่นยำ
วิธีนี้อาจมีประสิทธิภาพน้อยกว่าการใช้ standard gate ที่มีอยู่แล้ว หรือ custom Gate ที่ผู้ใช้นิยาม
สามารถนิยาม `gate` statement ด้วย class ที่เฉพาะเจาะจงได้ด้วยตนเอง

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### ตัวอย่าง: นิยาม built-in Gate ใหม่ในโปรแกรม OpenQASM 2
หาก argument `builtin=True` ถูกตั้งค่า custom Gate ไม่จำเป็นต้องมี definition ที่เกี่ยวข้อง